# **Model 2: Tyre Degradation Prediction**

## **Goal**
Predict lap time delta within a stint, relative to the stint's baseline lap.
This isolates tyre degradation from fuel burn and track evolution effects,
which already have their own features in the base table.

## **Target variable**
target = LapTime (seconds) - baseline_laptime_for_stint

baseline_laptime_for_stint = the first clean lap of that stint (TyreLife == 1,
green flag, not an out-lap). This gives us a per-lap "cost" attributable to
tyre wear, holding the rest of the lap's context roughly constant.

## **Scope and cleanups (same logic as notebook 07, now in a reusable function)**
- Race sessions only
- Red flag laps excluded (2024 Monaco GP Lap 1 contamination, same as before)
- is_out_lap derived the same way (PitOutTime.notna() & LapNumber != 1)
- prevstatus_* shift logic unchanged, same SC-start limitation applies
- Q1/Q2/Q3 columns dropped (100% null for race rows)

## **Model**
XGBoost, same baseline config as Model 1 (n_estimators=300, max_depth=6,
learning_rate=0.05, subsample=0.8, colsample_bytree=0.8). No reason to
switch algorithm family yet, feature engineering is the expected lever
here too based on Model 1's experience.

## **Plan for this notebook**
1. Load base table, build prepare_race_features(df) as a reusable function
2. Compute the stint-relative target variable
3. Grain and row-count checks after every major step
4. Baseline XGBoost with full feature set (reusing what worked for Model 1
   plus compound/temp interactions), see where MAE lands before deciding
   if domain-specific (Kimi) input is needed

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 100)

In [2]:
df_raw = pd.read_parquet('../data/processed/fastf1_ml_base.parquet')

print(f"Raw shape: {df_raw.shape}")
print(f"Seasons: {sorted(df_raw['Season'].unique())}")
print(f"SessionName values: {df_raw['SessionName'].unique()}")

Raw shape: (469012, 78)
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
SessionName values: ['Practice_1' 'Practice_2' 'Practice_3' 'Qualifying' 'Race' 'Sprint'
 'Sprint_Qualifying']


In [3]:
print(len(df_raw.columns))
print(sorted(df_raw.columns.tolist()))

78
['Abbreviation', 'AirTemp', 'BroadcastName', 'ClassifiedPosition', 'Compound', 'Compound_category', 'CountryCode', 'Deleted', 'DeletedReason', 'Driver', 'DriverId', 'DriverNumber', 'EventName', 'FastF1Generated', 'FinalPosition', 'FirstName', 'FreshTyre', 'FullName', 'GridPosition', 'HeadshotUrl', 'Humidity', 'IsAccurate', 'IsPersonalBest', 'LapNumber', 'LapStartDate', 'LapStartTime', 'LapTime', 'LapTime_seconds', 'Laps', 'LastName', 'PitInTime', 'PitOutTime', 'Points', 'Position', 'Pressure', 'Q1', 'Q2', 'Q3', 'Rainfall', 'ResultTime', 'Season', 'Sector1SessionTime', 'Sector1Time', 'Sector2SessionTime', 'Sector2Time', 'Sector3SessionTime', 'Sector3Time', 'SessionName', 'SpeedFL', 'SpeedI1', 'SpeedI2', 'SpeedST', 'Status', 'Stint', 'Team', 'TeamColor', 'TeamId', 'Time', 'TrackStatus', 'TrackTemp', 'TyreLife', 'WindDirection', 'WindSpeed', 'avg_speed', 'avg_throttle', 'brake_sample_pct', 'drs_active_pct', 'full_throttle_pct', 'max_rpm', 'max_speed', 'status_green', 'status_red_flag',

In [10]:
def prepare_race_features(df):
    """
    Reusable preprocessing function. Applies universal cleanups and features
    that are common across lap-time and tyre-degradation models.
    Model-specific target variable and model-specific features are added
    AFTER calling this function, not inside it.
    """
    df = df.copy()

    # ---- 1. Race-only scope ----
    df = df[df['SessionName'] == 'Race'].copy()

    # ---- 2. Red flag exclusion (contaminated LapTime, e.g. 2024 Monaco Lap 1) ----
    df = df[df['status_red_flag'] == False].copy()

    # ---- 3. Drop Q1/Q2/Q3 (100% null for race rows) ----
    df = df.drop(columns=['Q1', 'Q2', 'Q3'])

    # ---- 4. is_out_lap derivation ----
    # PitOutTime present AND LapNumber != 1 (exclude rolling/SC-start false positives)
    df['is_out_lap'] = df['PitOutTime'].notna() & (df['LapNumber'] != 1)

    # ---- 5. Sort for all groupby-based rolling/shift operations ----
    group_cols = ['Season', 'EventName', 'SessionName', 'Driver']
    df = df.sort_values(group_cols + ['LapNumber']).reset_index(drop=True)

    # ---- 6. prevstatus_* : shift each status flag forward by one lap within group ----
    status_cols = ['status_green', 'status_yellow', 'status_safety_car',
                   'status_red_flag', 'status_vsc', 'status_vsc_ending', 'status_unknown']

    for col in status_cols:
        shifted = df.groupby(group_cols)[col].shift(1)
        default_val = True if col == 'status_green' else False
        # Lap 1 default: green=True, everything else False
        df[f'prev{col}'] = np.where(shifted.isna(), default_val, shifted).astype(bool)

    # ---- 7. race_progress_pct ----
    total_laps_per_race = df.groupby(['Season', 'EventName'])['LapNumber'].transform('max')
    df['race_progress_pct'] = df['LapNumber'] / total_laps_per_race

    # ---- 8. fuel_load_proxy / fuel_load_pct ----
    laps_remaining = total_laps_per_race - df['LapNumber']
    df['fuel_load_proxy'] = laps_remaining
    df['fuel_load_pct'] = laps_remaining / total_laps_per_race

   # ---- 9. cumulative_green_laps (track evolution proxy) ----
    shifted_green = df.groupby(['Season', 'EventName'])['status_green'].shift(1)
    shifted_green_filled = np.where(shifted_green.isna(), True, shifted_green).astype(bool)
    df['_shifted_green_temp'] = shifted_green_filled
    df['cumulative_green_laps'] = (
        df.groupby(['Season', 'EventName'])['_shifted_green_temp']
        .transform(lambda x: x.astype(int).cumsum())
    )
    df = df.drop(columns=['_shifted_green_temp'])
    df['green_laps_sqrt'] = np.sqrt(df['cumulative_green_laps'])
    # ---- 10. TyreLife derived features ----
    df['tyrelife_squared'] = df['TyreLife'] ** 2
    df['tyrelife_sqrt'] = np.sqrt(df['TyreLife'])
    df['tyrelife_log1p'] = np.log1p(df['TyreLife'])

    # stint_lap_number: lap count within current stint (should mirror TyreLife+1 pattern,
    # kept separate since TyreLife semantics were verified via Kimi specifically for this)
    df['stint_lap_number'] = df.groupby(group_cols + ['Stint']).cumcount() + 1

    # ---- 11. Compound_category one-hot × TyreLife, × TrackTemp interactions ----
    compound_dummies = pd.get_dummies(df['Compound_category'], prefix='compound')
    for col in compound_dummies.columns:
        df[f'{col}_x_tyrelife'] = compound_dummies[col] * df['TyreLife']
        df[f'{col}_x_tracktemp'] = compound_dummies[col] * df['TrackTemp']
    df = pd.concat([df, compound_dummies], axis=1)

    # ---- 12. track_temp_minus_airtemp ----
    df['track_temp_minus_airtemp'] = df['TrackTemp'] - df['AirTemp']

    # ---- 13. Clean lap flag (used for driver-form rolling features) ----
    df['is_clean_lap'] = (
        df['status_green'] & (~df['is_out_lap']) & (~df['status_red_flag'])
    )

    # ---- 14. Driver rolling pace features (leakage-safe: shift(1) before rolling) ----
    df['driver_prev_lap_time'] = df.groupby(group_cols)['LapTime_seconds'].shift(1)

    clean_laptime = df['LapTime_seconds'].where(df['is_clean_lap'])
    df['_clean_laptime_shifted'] = df.groupby(group_cols)['LapTime_seconds'].shift(1).where(
        df.groupby(group_cols)['is_clean_lap'].shift(1)
    )

    df['driver_rolling_median_3'] = (
        df.groupby(group_cols)['_clean_laptime_shifted']
        .transform(lambda x: x.rolling(3, min_periods=1).median())
    )
    df['driver_best_lap_so_far'] = (
        df.groupby(group_cols)['_clean_laptime_shifted']
        .transform(lambda x: x.expanding(min_periods=1).min())
    )
    df['driver_pace_index'] = df['driver_prev_lap_time'] - df['driver_best_lap_so_far']
    df = df.drop(columns=['_clean_laptime_shifted'])

    # ---- 15. Strategic context ----
    df['position_prev_lap'] = df.groupby(group_cols)['Position'].shift(1)
    df['grid_delta'] = df['GridPosition'] - df['Position']
    df['is_race_leader'] = (df['Position'] == 1)

   # ---- 16. SC/VSC counters (fixed version) ----
    df['laps_since_sc_started'] = (
        df.groupby(['Season', 'EventName'])['status_safety_car']
        .transform(lambda x: x.groupby((~x).cumsum()).cumcount())
    )
    df['laps_since_vsc_started'] = (
        df.groupby(['Season', 'EventName'])['status_vsc']
        .transform(lambda x: x.groupby((~x).cumsum()).cumcount())
    )

    shifted_sc = df.groupby(['Season', 'EventName'])['status_safety_car'].shift(1)
    shifted_sc_filled = np.where(shifted_sc.isna(), False, shifted_sc).astype(bool)
    df['_shifted_sc_temp'] = shifted_sc_filled
    df['sc_deployment_count'] = (
        df.groupby(['Season', 'EventName'])
        .apply(lambda g: (g['status_safety_car'] & ~g['_shifted_sc_temp']).cumsum(), include_groups=False)
        .reset_index(level=[0,1], drop=True)
    )
    df = df.drop(columns=['_shifted_sc_temp'])

    return df

In [11]:
print(f"Before: {df_raw.shape}")

df_prepared = prepare_race_features(df_raw)

print(f"After: {df_prepared.shape}")
print(f"Columns added: {df_prepared.shape[1] - df_raw.shape[1]}")
print(f"\nRace rows only check: {df_prepared['SessionName'].unique()}")
print(f"Red flag rows remaining: {df_prepared['status_red_flag'].sum()}")
print(f"\nNulls in key new features:")
check_cols = ['TyreLife', 'tyrelife_sqrt', 'driver_rolling_median_3', 'cumulative_green_laps']
print(df_prepared[check_cols].isnull().sum())

Before: (469012, 78)
After: (188056, 122)
Columns added: 44

Race rows only check: ['Race']
Red flag rows remaining: 0

Nulls in key new features:
TyreLife                   1351
tyrelife_sqrt              1351
driver_rolling_median_3    8226
cumulative_green_laps         0
dtype: int64


In [12]:
null_tyrelife = df_prepared[df_prepared['TyreLife'].isnull()]
print(null_tyrelife[['Season', 'EventName', 'LapNumber']].drop_duplicates().shape)
print(null_tyrelife['Season'].value_counts())
print(null_tyrelife[['Season','EventName']].drop_duplicates().head(10))

(243, 3)
Season
2022    510
2025    434
2018    303
2020     69
2023     35
Name: count, dtype: int64
       Season              EventName
943      2018  Australian_Grand_Prix
1883     2018    Austrian_Grand_Prix
3178     2018  Azerbaijan_Grand_Prix
3975     2018     Bahrain_Grand_Prix
5578     2018     Belgian_Grand_Prix
6984     2018     British_Grand_Prix
7886     2018    Canadian_Grand_Prix
9105     2018     Chinese_Grand_Prix
10324    2018      French_Grand_Prix
11140    2018      German_Grand_Prix


In [14]:
# check if these are mostly retirement/DNF laps
null_tyrelife = df_prepared[df_prepared['TyreLife'].isnull()]
print(null_tyrelife['Status'].value_counts())
print(null_tyrelife['ClassifiedPosition'].value_counts())

# also check LapNumber distribution - end of race laps (retirements) vs random
print(null_tyrelife.groupby(['Season','EventName'])['LapNumber'].max().value_counts().head(10))

Status
Finished            692
+1 Lap              470
Retired              47
Lapped               46
+2 Laps              17
Undertray            13
Accident             13
Engine                7
Power Unit            7
Collision             6
Mechanical            5
Brakes                4
Wheel                 3
Gearbox               3
Puncture              2
Exhaust               2
Hydraulics            2
Fuel pressure         2
Electrical            2
Collision damage      2
Steering              1
Tyre                  1
Oil leak              1
Power loss            1
+3 Laps               1
Turbo                 1
Name: count, dtype: int64
ClassifiedPosition
R     123
15    101
14    101
16     90
8      75
9      74
10     73
5      73
2      72
7      72
6      71
3      64
12     63
1      63
17     55
11     51
13     47
4      43
18     21
19     18
20      1
Name: count, dtype: int64
LapNumber
2.0     14
44.0     2
51.0     1
69.0     1
1.0      1
71.0     1
63.0     1
7

In [15]:
lap57_nulls = df_prepared[(df_prepared['driver_rolling_median_3'].isnull()) & (df_prepared['LapNumber'] == 57)]
print(lap57_nulls[['Season','EventName','Driver']].drop_duplicates())

        Season           EventName Driver
11261     2018   German_Grand_Prix    BOT
11328     2018   German_Grand_Prix    ERI
11461     2018   German_Grand_Prix    GRO
11528     2018   German_Grand_Prix    HAM
11595     2018   German_Grand_Prix    HAR
...        ...                 ...    ...
171922    2025    Dutch_Grand_Prix    SAI
171994    2025    Dutch_Grand_Prix    STR
172066    2025    Dutch_Grand_Prix    TSU
172138    2025    Dutch_Grand_Prix    VER
184870    2025  Spanish_Grand_Prix    COL

[126 rows x 3 columns]


In [13]:
null_pace = df_prepared[df_prepared['driver_rolling_median_3'].isnull()]
print(null_pace['LapNumber'].value_counts().head(10))

LapNumber
1.0     3341
5.0      441
6.0      313
2.0      280
3.0      207
4.0      163
57.0     126
14.0     111
7.0      111
29.0      97
Name: count, dtype: int64


In [16]:
print(f"Before dropping TyreLife nulls: {df_prepared.shape}")
df_prepared = df_prepared[df_prepared['TyreLife'].notna()].copy()
print(f"After: {df_prepared.shape}")

Before dropping TyreLife nulls: (188056, 122)
After: (186705, 122)


## Data quality findings (notebook 08)

Two null patterns found after running prepare_race_features():

1. TyreLife nulls (1351 rows, 0.7% of race data): mostly concentrated at
   LapNumber 1-2 across many races. Same root cause family as the already
   known SC-start / rolling-start limitation, TyreLife tracking doesn't
   properly initialize on some rolling starts. These rows are dropped since
   TyreLife is a core feature for degradation modeling, can't be imputed
   meaningfully. Dropped: 188056 -> 186705 rows.

2. driver_rolling_median_3 nulls (8226 rows): mostly Lap 1 (no prior clean
   lap exists yet) plus a smaller cluster at high lap numbers in a few wet
   or chaotic races (2018 German GP, 2025 Dutch GP, 2025 Spanish GP), where
   no clean green-flag lap existed for some drivers for most of the race.
   This is genuine data, not a bug. XGBoost handles NaN natively in splits,
   so these rows are kept as-is, same approach as Model 1.

**Target Variable and Baseline**

In [19]:
# ---- Target variable: lap time delta relative to stint's first clean lap ----

group_stint_cols = ['Season', 'EventName', 'SessionName', 'Driver', 'Stint']

def get_stint_baseline_v2(group):
    clean_first = group[
        (~group['is_out_lap']) &
        (group['status_green']) &
        (group['LapTime_seconds'].notna())
    ].sort_values('TyreLife')

    if len(clean_first) > 0:
        return clean_first['LapTime_seconds'].iloc[0]
    else:
        # true fallback: no clean lap exists anywhere in this stint (e.g. entire stint under SC/VSC)
        valid = group[group['LapTime_seconds'].notna()].sort_values('TyreLife')
        if len(valid) > 0:
            return valid['LapTime_seconds'].iloc[0]
        else:
            return np.nan

stint_baselines_v2 = (
    df_prepared.groupby(group_stint_cols)
    .apply(get_stint_baseline_v2, include_groups=False)
    .rename('stint_baseline_laptime_v2')
)

df_prepared = df_prepared.drop(columns=['stint_baseline_laptime', 'tyre_deg_target']).merge(
    stint_baselines_v2, on=group_stint_cols, how='left'
)
df_prepared = df_prepared.rename(columns={'stint_baseline_laptime_v2': 'stint_baseline_laptime'})
df_prepared['tyre_deg_target'] = df_prepared['LapTime_seconds'] - df_prepared['stint_baseline_laptime']

print(f"Nulls in stint_baseline_laptime: {df_prepared['stint_baseline_laptime'].isnull().sum()}")
print(f"Nulls in tyre_deg_target: {df_prepared['tyre_deg_target'].isnull().sum()}")
print(f"\ntyre_deg_target distribution:")
print(df_prepared['tyre_deg_target'].describe())

baseline_laps_v2 = df_prepared[df_prepared['LapTime_seconds'] == df_prepared['stint_baseline_laptime']]
print(f"\nBaseline lap target mean (should be ~0): {baseline_laps_v2['tyre_deg_target'].mean():.4f}")

Nulls in stint_baseline_laptime: 236
Nulls in tyre_deg_target: 2865

tyre_deg_target distribution:
count    183840.000000
mean         -5.438019
std          14.709229
min         -97.788000
25%          -8.297000
50%          -0.757000
75%           0.351000
max          74.828000
Name: tyre_deg_target, dtype: float64

Baseline lap target mean (should be ~0): 0.0000


In [20]:
target_by_stint_lap = (
    df_prepared.groupby('stint_lap_number')['tyre_deg_target']
    .agg(['mean', 'count'])
    .reset_index()
)

print(target_by_stint_lap.head(20))

    stint_lap_number       mean  count
0                1.0  10.753118   8262
1                2.0  -1.083119   8068
2                3.0  -2.264760   7938
3                4.0  -3.690962   7885
4                5.0  -4.795124   7955
5                6.0  -5.806098   7874
6                7.0  -6.115463   7791
7                8.0  -6.555270   7628
8                9.0  -6.676245   7463
9               10.0  -6.588744   7301
10              11.0  -6.966983   7088
11              12.0  -6.848269   6865
12              13.0  -6.782062   6587
13              14.0  -6.721881   6322
14              15.0  -7.000774   6051
15              16.0  -7.007835   5807
16              17.0  -7.086692   5538
17              18.0  -7.059186   5250
18              19.0  -7.283489   4902
19              20.0  -7.043425   4589


In [21]:
# Out-laps ka target meaningless hai degradation ke liye, unhe null kar dete hain
df_prepared.loc[df_prepared['is_out_lap'], 'tyre_deg_target'] = np.nan

print(f"Nulls in tyre_deg_target after excluding out-laps: {df_prepared['tyre_deg_target'].isnull().sum()}")

# Recheck the pattern by stint_lap_number
target_by_stint_lap_v2 = (
    df_prepared.groupby('stint_lap_number')['tyre_deg_target']
    .agg(['mean', 'count'])
    .reset_index()
)
print(target_by_stint_lap_v2.head(10))

Nulls in tyre_deg_target after excluding out-laps: 7945
   stint_lap_number      mean  count
0               1.0  0.279197   3279
1               2.0 -1.094368   8061
2               3.0 -2.273932   7930
3               4.0 -3.693589   7883
4               5.0 -4.795124   7955
5               6.0 -5.832909   7867
6               7.0 -6.115463   7791
7               8.0 -6.555270   7628
8               9.0 -6.675906   7462
9              10.0 -6.591264   7299


## Target variable finalized: tyre_deg_target

tyre_deg_target = LapTime_seconds - stint_baseline_laptime

stint_baseline_laptime = the first non-out-lap, green-flag lap of the stint
(not necessarily TyreLife==1, since TyreLife==1 laps after a pit stop are
always the out-lap itself).

Out-laps are excluded from the target (set to NaN), since their slowness
comes from pit lane speed limits and cold tyres, not degradation.

Sanity check: grouping by stint_lap_number shows the target starting near
zero at the baseline lap and drifting negative through laps 2-9 (fuel burn
making the car lighter and faster, outweighing early tyre wear), then
plateauing around -6.5 to -7 seconds from lap 9 onward (tyre degradation
starting to offset the fuel effect). This is expected real-world behavior,
not a bug. The model will use fuel_load_proxy as a separate feature to
help disentangle the two effects.

Rows dropped: out-lap rows now have null target (7945 rows), excluded
before training.

**Training-Ready Dataframe**
___

In [22]:
df_model = df_prepared[df_prepared['tyre_deg_target'].notna()].copy()
print(f"Final shape for modeling: {df_model.shape}")

Final shape for modeling: (178760, 124)


**Feature selection (From Model 1 and Kimi Suggestion)**
___

In [29]:
feature_columns = [
    'TyreLife', 'tyrelife_squared', 'tyrelife_sqrt', 'tyrelife_log1p', 'stint_lap_number',

    'compound_HARD', 'compound_MEDIUM', 'compound_SOFT',  
    'compound_HARD_x_tyrelife', 'compound_MEDIUM_x_tyrelife', 'compound_SOFT_x_tyrelife',
    'compound_HARD_x_tracktemp', 'compound_MEDIUM_x_tracktemp', 'compound_SOFT_x_tracktemp',

    'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp',
    'WindDirection', 'WindSpeed', 'track_temp_minus_airtemp',

    'fuel_load_proxy', 'fuel_load_pct', 'race_progress_pct',
    'cumulative_green_laps', 'green_laps_sqrt',

    'driver_prev_lap_time', 'driver_rolling_median_3',
    'driver_best_lap_so_far', 'driver_pace_index',

    'position_prev_lap', 'grid_delta', 'is_race_leader',

    'laps_since_sc_started', 'laps_since_vsc_started', 'sc_deployment_count',

    'prevstatus_green', 'prevstatus_yellow', 'prevstatus_safety_car',
    'prevstatus_vsc', 'prevstatus_vsc_ending',

    'Driver', 'Team', 'EventName',
]

missing = [c for c in feature_columns if c not in df_model.columns]
print(f"Missing columns: {missing}")

Missing columns: []


In [25]:
print(df_model['Compound_category'].unique())
print([c for c in df_model.columns if 'compound' in c.lower()])

['MEDIUM' 'HARD' 'SOFT' 'INTERMEDIATE' 'WET' 'UNKNOWN']
['Compound', 'Compound_category', 'compound_HARD_x_tyrelife', 'compound_HARD_x_tracktemp', 'compound_INTERMEDIATE_x_tyrelife', 'compound_INTERMEDIATE_x_tracktemp', 'compound_MEDIUM_x_tyrelife', 'compound_MEDIUM_x_tracktemp', 'compound_SOFT_x_tyrelife', 'compound_SOFT_x_tracktemp', 'compound_UNKNOWN_x_tyrelife', 'compound_UNKNOWN_x_tracktemp', 'compound_WET_x_tyrelife', 'compound_WET_x_tracktemp', 'compound_HARD', 'compound_INTERMEDIATE', 'compound_MEDIUM', 'compound_SOFT', 'compound_UNKNOWN', 'compound_WET']


In [30]:
feature_columns = [
    # Tyre core features
    'TyreLife', 'tyrelife_squared', 'tyrelife_sqrt', 'tyrelife_log1p', 'stint_lap_number',

    # Compound one-hot + interactions (all categories, including wet-weather ones)
    'compound_HARD', 'compound_MEDIUM', 'compound_SOFT',
    'compound_INTERMEDIATE', 'compound_WET', 'compound_UNKNOWN',
    'compound_HARD_x_tyrelife', 'compound_MEDIUM_x_tyrelife', 'compound_SOFT_x_tyrelife',
    'compound_INTERMEDIATE_x_tyrelife', 'compound_WET_x_tyrelife', 'compound_UNKNOWN_x_tyrelife',
    'compound_HARD_x_tracktemp', 'compound_MEDIUM_x_tracktemp', 'compound_SOFT_x_tracktemp',
    'compound_INTERMEDIATE_x_tracktemp', 'compound_WET_x_tracktemp', 'compound_UNKNOWN_x_tracktemp',

    # Weather
    'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp',
    'WindDirection', 'WindSpeed', 'track_temp_minus_airtemp',

    # Fuel / race progress / track evolution
    'fuel_load_proxy', 'fuel_load_pct', 'race_progress_pct',
    'cumulative_green_laps', 'green_laps_sqrt',

    # Driver pace / form
    'driver_prev_lap_time', 'driver_rolling_median_3',
    'driver_best_lap_so_far', 'driver_pace_index',

    # Strategic context
    'position_prev_lap', 'grid_delta', 'is_race_leader',

    # SC/VSC (untested for this target, worth trying fresh per your notes)
    'laps_since_sc_started', 'laps_since_vsc_started', 'sc_deployment_count',

    # Track status (previous lap, leakage-safe)
    'prevstatus_green', 'prevstatus_yellow', 'prevstatus_safety_car',
    'prevstatus_vsc', 'prevstatus_vsc_ending',

    # Categorical (to be label encoded)
    'Driver', 'Team', 'EventName',
]

missing = [c for c in feature_columns if c not in df_model.columns]
print(f"Missing columns: {missing}")
print(f"Total features: {len(feature_columns)}")

Missing columns: []
Total features: 54


**Label Encoding**
___

In [31]:
categorical_cols = ['Driver', 'Team', 'EventName']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col + '_encoded'] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le


feature_columns_final = [c for c in feature_columns if c not in categorical_cols] + \
                         [c + '_encoded' for c in categorical_cols]

print(f"Final feature count: {len(feature_columns_final)}")
print(feature_columns_final)

Final feature count: 54
['TyreLife', 'tyrelife_squared', 'tyrelife_sqrt', 'tyrelife_log1p', 'stint_lap_number', 'compound_HARD', 'compound_MEDIUM', 'compound_SOFT', 'compound_INTERMEDIATE', 'compound_WET', 'compound_UNKNOWN', 'compound_HARD_x_tyrelife', 'compound_MEDIUM_x_tyrelife', 'compound_SOFT_x_tyrelife', 'compound_INTERMEDIATE_x_tyrelife', 'compound_WET_x_tyrelife', 'compound_UNKNOWN_x_tyrelife', 'compound_HARD_x_tracktemp', 'compound_MEDIUM_x_tracktemp', 'compound_SOFT_x_tracktemp', 'compound_INTERMEDIATE_x_tracktemp', 'compound_WET_x_tracktemp', 'compound_UNKNOWN_x_tracktemp', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed', 'track_temp_minus_airtemp', 'fuel_load_proxy', 'fuel_load_pct', 'race_progress_pct', 'cumulative_green_laps', 'green_laps_sqrt', 'driver_prev_lap_time', 'driver_rolling_median_3', 'driver_best_lap_so_far', 'driver_pace_index', 'position_prev_lap', 'grid_delta', 'is_race_leader', 'laps_since_sc_started', 'laps_since_v

**Train/Test Split**
___

In [33]:
train_df = df_model[df_model['Season'] <= 2024].copy()
test_df = df_model[df_model['Season'] == 2025].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

X_train = train_df[feature_columns_final]
y_train = train_df['tyre_deg_target']
X_test = test_df[feature_columns_final]
y_test = test_df['tyre_deg_target']

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Train shape: (153613, 127)
Test shape: (25147, 127)

X_train: (153613, 54), y_train: (153613,)
X_test: (25147, 54), y_test: (25147,)


### **XGBoost Baseline**
___

In [36]:
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.3f}s")
print(f"RMSE: {rmse:.3f}s")
print(f"R2: {r2:.3f}")

MAE: 7.228s
RMSE: 12.087s
R2: 0.326


**Too WEAK !!!**
___

**Feature Importance**

In [39]:
importance_df = pd.DataFrame({
    'feature': feature_columns_final,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df.head(20))

                              feature  importance
46                   prevstatus_green    0.114637
43              laps_since_sc_started    0.110417
21           compound_WET_x_tracktemp    0.101713
48              prevstatus_safety_car    0.046352
45                sc_deployment_count    0.041071
44             laps_since_vsc_started    0.039525
15            compound_WET_x_tyrelife    0.035426
33                  race_progress_pct    0.034923
32                      fuel_load_pct    0.024756
4                    stint_lap_number    0.024587
39                  driver_pace_index    0.023259
53                  EventName_encoded    0.021973
23                            AirTemp    0.021519
37            driver_rolling_median_3    0.020420
25                           Pressure    0.017745
24                           Humidity    0.016934
47                  prevstatus_yellow    0.016769
17          compound_HARD_x_tracktemp    0.016413
20  compound_INTERMEDIATE_x_tracktemp    0.014674


In [40]:
clean_lap_mask = df_model['status_green'] == True
print(f"Total rows: {len(df_model)}")
print(f"Clean (green flag) rows: {clean_lap_mask.sum()}")
print(f"Non-green rows being dropped: {(~clean_lap_mask).sum()}")
print(f"Percentage dropped: {(~clean_lap_mask).mean()*100:.1f}%")

Total rows: 178760
Clean (green flag) rows: 172315
Non-green rows being dropped: 6445
Percentage dropped: 3.6%


In [41]:
df_model_clean = df_model[df_model['status_green'] == True].copy()
print(f"Clean training data shape: {df_model_clean.shape}")

train_df2 = df_model_clean[df_model_clean['Season'] <= 2024].copy()
test_df2 = df_model_clean[df_model_clean['Season'] == 2025].copy()

X_train2 = train_df2[feature_columns_final]
y_train2 = train_df2['tyre_deg_target']
X_test2 = test_df2[feature_columns_final]
y_test2 = test_df2['tyre_deg_target']

print(f"X_train2: {X_train2.shape}, X_test2: {X_test2.shape}")

model2 = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)

mae2 = mean_absolute_error(y_test2, y_pred2)
rmse2 = np.sqrt(mean_squared_error(y_test2, y_pred2))
r2_2 = r2_score(y_test2, y_pred2)

print(f"\nMAE: {mae2:.3f}s")
print(f"RMSE: {rmse2:.3f}s")
print(f"R2: {r2_2:.3f}")

importance_df2 = pd.DataFrame({
    'feature': feature_columns_final,
    'importance': model2.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\nTop 15 features:")
print(importance_df2.head(15))

Clean training data shape: (172315, 127)
X_train2: (147946, 54), X_test2: (24369, 54)

MAE: 7.232s
RMSE: 11.901s
R2: 0.288

Top 15 features:
                     feature  importance
21  compound_WET_x_tracktemp    0.106173
43     laps_since_sc_started    0.095955
46          prevstatus_green    0.062475
45       sc_deployment_count    0.047672
48     prevstatus_safety_car    0.036974
32             fuel_load_pct    0.036386
9               compound_WET    0.031560
33         race_progress_pct    0.031355
53         EventName_encoded    0.028069
15   compound_WET_x_tyrelife    0.027124
24                  Humidity    0.023794
39         driver_pace_index    0.023220
23                   AirTemp    0.023185
44    laps_since_vsc_started    0.022968
37   driver_rolling_median_3    0.021349


## BASELINE FIX
___

In [42]:
def get_baseline_v3(group):
    clean = group[(~group['is_out_lap']) & (group['status_green']) & (group['LapTime_seconds'].notna())]
    clean_sorted = clean.sort_values('stint_lap_number')
    first_3 = clean_sorted.head(3)
    if len(first_3) > 0:
        return first_3['LapTime_seconds'].min()
    else:
        valid = group[group['LapTime_seconds'].notna()].sort_values('TyreLife')
        return valid['LapTime_seconds'].iloc[0] if len(valid) > 0 else np.nan

stint_baselines_v3 = (
    df_prepared.groupby(group_stint_cols)
    .apply(get_baseline_v3, include_groups=False)
    .rename('stint_baseline_v3')
)

df_prepared = df_prepared.merge(stint_baselines_v3, on=group_stint_cols, how='left')
df_prepared['tyre_deg_target_v3'] = df_prepared['LapTime_seconds'] - df_prepared['stint_baseline_v3']
df_prepared.loc[df_prepared['is_out_lap'], 'tyre_deg_target_v3'] = np.nan

print(f"Nulls: {df_prepared['tyre_deg_target_v3'].isnull().sum()}")
print(df_prepared['tyre_deg_target_v3'].describe())

target_by_lap_v3 = df_prepared.groupby('stint_lap_number')['tyre_deg_target_v3'].mean().head(15)
print(f"\nTarget by stint_lap_number:")
print(target_by_lap_v3)

Nulls: 7945
count    178760.000000
mean          1.409057
std           7.670612
min         -56.727000
25%          -0.651000
50%           0.056000
75%           0.841000
max          95.922000
Name: tyre_deg_target_v3, dtype: float64

Target by stint_lap_number:
stint_lap_number
1.0     10.276705
2.0      5.655020
3.0      4.545701
4.0      3.301290
5.0      2.773656
6.0      1.825871
7.0      1.518864
8.0      1.065970
9.0      0.880536
10.0     0.987800
11.0     0.648722
12.0     0.655993
13.0     0.606525
14.0     0.657245
15.0     0.493370
Name: tyre_deg_target_v3, dtype: float64


In [44]:
group_driver_race = ['Season', 'EventName', 'SessionName', 'Driver']

df_prepared = df_prepared.sort_values(group_driver_race + ['LapNumber']).reset_index(drop=True)

df_prepared['sc_or_vsc_active'] = df_prepared['prevstatus_safety_car'] | df_prepared['prevstatus_vsc']

df_prepared['green_period_id'] = (
    df_prepared.groupby(group_driver_race)['sc_or_vsc_active']
    .transform(lambda x: (~x).ne((~x).shift()).cumsum())
)

df_prepared['laps_since_green_resumed'] = (
    df_prepared.groupby(group_driver_race + ['green_period_id']).cumcount()
)
df_prepared['laps_since_green_resumed'] = df_prepared['laps_since_green_resumed'].clip(0, 10)
df_prepared['sc_recovery_decay'] = (1 - df_prepared['laps_since_green_resumed'] / 10).clip(0, 1)
df_prepared['is_restart_lap'] = (df_prepared['laps_since_green_resumed'] == 1)

print(df_prepared[['laps_since_green_resumed', 'sc_recovery_decay', 'is_restart_lap']].describe())

       laps_since_green_resumed  sc_recovery_decay
count             186705.000000      186705.000000
mean                   7.821628           0.217837
std                    3.434860           0.343486
min                    0.000000           0.000000
25%                    6.000000           0.000000
50%                   10.000000           0.000000
75%                   10.000000           0.400000
max                   10.000000           1.000000


In [48]:
df_prepared['is_warmup_phase'] = df_prepared['stint_lap_number'] <= 3
df_prepared['warmup_laps_remaining'] = (3 - df_prepared['stint_lap_number']).clip(0, 3)
df_prepared['stint_phase'] = pd.cut(
    df_prepared['stint_lap_number'],
    bins=[0, 3, 10, 20, 999],
    labels=['warmup', 'early', 'mid', 'late']
)

print(df_prepared['stint_phase'].value_counts())

stint_phase
mid       59273
early     54575
late      46927
warmup    25930
Name: count, dtype: int64


In [46]:
for compound in ['SOFT', 'MEDIUM', 'HARD']:
    mask = (df_prepared['Compound_category'] == compound).astype(int)
    df_prepared[f'{compound}_x_stint_lap'] = mask * df_prepared['stint_lap_number']
    df_prepared[f'{compound}_x_stint_lap_sq'] = mask * df_prepared['tyrelife_squared']

print([c for c in df_prepared.columns if '_x_stint_lap' in c])

['SOFT_x_stint_lap', 'SOFT_x_stint_lap_sq', 'MEDIUM_x_stint_lap', 'MEDIUM_x_stint_lap_sq', 'HARD_x_stint_lap', 'HARD_x_stint_lap_sq']


In [47]:
target_by_lap_extended = df_prepared.groupby('stint_lap_number')['tyre_deg_target_v3'].agg(['mean', 'count']).head(40)
print(target_by_lap_extended)

                       mean  count
stint_lap_number                  
1.0               10.276705   3279
2.0                5.655020   8061
3.0                4.545701   7930
4.0                3.301290   7883
5.0                2.773656   7955
6.0                1.825871   7867
7.0                1.518864   7791
8.0                1.065970   7628
9.0                0.880536   7462
10.0               0.987800   7299
11.0               0.648722   7087
12.0               0.655993   6863
13.0               0.606525   6585
14.0               0.657245   6322
15.0               0.493370   6051
16.0               0.470938   5804
17.0               0.394585   5536
18.0               0.416406   5246
19.0               0.170011   4899
20.0               0.329477   4587
21.0               0.380132   4284
22.0               0.417154   3989
23.0               0.350234   3743
24.0               0.563703   3453
25.0               0.276427   3166
26.0               0.155830   2878
27.0               0

In [49]:
df_dry = df_prepared[df_prepared['Compound_category'].isin(['HARD', 'MEDIUM', 'SOFT'])].copy()
df_wet = df_prepared[df_prepared['Compound_category'].isin(['INTERMEDIATE', 'WET'])].copy()

print(f"Dry rows: {df_dry.shape[0]}")
print(f"Wet rows: {df_wet.shape[0]}")
print(f"Unknown/other rows: {df_prepared.shape[0] - df_dry.shape[0] - df_wet.shape[0]}")

# dry compounds pe hi degradation curve check karte hain
dry_target_by_lap = df_dry.groupby('stint_lap_number')['tyre_deg_target_v3'].agg(['mean','count']).head(30)
print(dry_target_by_lap)

Dry rows: 176573
Wet rows: 9993
Unknown/other rows: 139
                      mean  count
stint_lap_number                 
1.0               9.638125   2982
2.0               5.343641   7510
3.0               4.211530   7400
4.0               3.032278   7384
5.0               2.706532   7464
6.0               1.765721   7401
7.0               1.532853   7341
8.0               1.064384   7194
9.0               0.917534   7080
10.0              1.030849   6931
11.0              0.780688   6712
12.0              0.775609   6512
13.0              0.724471   6259
14.0              0.815633   6021
15.0              0.654974   5755
16.0              0.679526   5515
17.0              0.608495   5252
18.0              0.639621   4975
19.0              0.408268   4651
20.0              0.589483   4356
21.0              0.608116   4063
22.0              0.676322   3796
23.0              0.677235   3555
24.0              0.885086   3283
25.0              0.408711   3011
26.0              0.240075

In [50]:
# stint_phase ko one-hot karte hain (categorical hai abhi)
stint_phase_dummies = pd.get_dummies(df_dry['stint_phase'], prefix='phase')
df_dry = pd.concat([df_dry, stint_phase_dummies], axis=1)

feature_columns_v2 = [
    # Tyre core
    'TyreLife', 'tyrelife_squared', 'tyrelife_sqrt', 'tyrelife_log1p', 'stint_lap_number',

    # Compound one-hot + interactions (dry only, so just 3)
    'compound_HARD', 'compound_MEDIUM', 'compound_SOFT',
    'compound_HARD_x_tyrelife', 'compound_MEDIUM_x_tyrelife', 'compound_SOFT_x_tyrelife',
    'compound_HARD_x_tracktemp', 'compound_MEDIUM_x_tracktemp', 'compound_SOFT_x_tracktemp',

    # New: Kimi's compound x stint_lap interactions
    'SOFT_x_stint_lap', 'SOFT_x_stint_lap_sq',
    'MEDIUM_x_stint_lap', 'MEDIUM_x_stint_lap_sq',
    'HARD_x_stint_lap', 'HARD_x_stint_lap_sq',

    # New: warmup phase features
    'is_warmup_phase', 'warmup_laps_remaining',
    'phase_warmup', 'phase_early', 'phase_mid', 'phase_late',

    # New: SC/VSC recovery decay
    'laps_since_green_resumed', 'sc_recovery_decay', 'is_restart_lap',

    # Weather
    'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp',
    'WindDirection', 'WindSpeed', 'track_temp_minus_airtemp',

    # Fuel / race progress / track evolution
    'fuel_load_proxy', 'fuel_load_pct', 'race_progress_pct',
    'cumulative_green_laps', 'green_laps_sqrt',

    # Driver pace/form
    'driver_prev_lap_time', 'driver_rolling_median_3',
    'driver_best_lap_so_far', 'driver_pace_index',

    # Strategic
    'position_prev_lap', 'grid_delta', 'is_race_leader',

    # SC/VSC counters (old ones, testing if still needed alongside new decay features)
    'laps_since_sc_started', 'laps_since_vsc_started', 'sc_deployment_count',

    'prevstatus_green', 'prevstatus_yellow', 'prevstatus_safety_car',
    'prevstatus_vsc', 'prevstatus_vsc_ending',

    'Driver', 'Team', 'EventName',
]

missing = [c for c in feature_columns_v2 if c not in df_dry.columns]
print(f"Missing: {missing}")
print(f"Total features: {len(feature_columns_v2)}")

Missing: []
Total features: 60


In [51]:
categorical_cols = ['Driver', 'Team', 'EventName']
label_encoders_dry = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_dry[col + '_encoded'] = le.fit_transform(df_dry[col].astype(str))
    label_encoders_dry[col] = le

feature_columns_v2_final = [c for c in feature_columns_v2 if c not in categorical_cols] + \
                           [c + '_encoded' for c in categorical_cols]

# only keep rows with a valid target
df_dry_model = df_dry[df_dry['tyre_deg_target_v3'].notna()].copy()
print(f"Dry model data shape: {df_dry_model.shape}")

train_dry = df_dry_model[df_dry_model['Season'] <= 2024].copy()
test_dry = df_dry_model[df_dry_model['Season'] == 2025].copy()

X_train_dry = train_dry[feature_columns_v2_final]
y_train_dry = train_dry['tyre_deg_target_v3']
X_test_dry = test_dry[feature_columns_v2_final]
y_test_dry = test_dry['tyre_deg_target_v3']

print(f"X_train: {X_train_dry.shape}, X_test: {X_test_dry.shape}")

Dry model data shape: (169378, 147)
X_train: (145574, 60), X_test: (23804, 60)


**Training**
___

In [53]:
model_dry = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_dry.fit(X_train_dry, y_train_dry)
y_pred_dry = model_dry.predict(X_test_dry)

mae_dry = mean_absolute_error(y_test_dry, y_pred_dry)
rmse_dry = np.sqrt(mean_squared_error(y_test_dry, y_pred_dry))
r2_dry = r2_score(y_test_dry, y_pred_dry)

print(f"MAE: {mae_dry:.3f}s")
print(f"RMSE: {rmse_dry:.3f}s")
print(f"R2: {r2_dry:.3f}")

importance_dry = pd.DataFrame({
    'feature': feature_columns_v2_final,
    'importance': model_dry.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 20 features:")
print(importance_dry.head(20))

MAE: 1.451s
RMSE: 3.299s
R2: 0.800

Top 20 features:
                     feature  importance
54     prevstatus_safety_car    0.290054
49     laps_since_sc_started    0.198114
52          prevstatus_green    0.105185
50    laps_since_vsc_started    0.078141
27         sc_recovery_decay    0.024035
55            prevstatus_vsc    0.022840
26  laps_since_green_resumed    0.018736
53         prevstatus_yellow    0.017506
56     prevstatus_vsc_ending    0.016816
22              phase_warmup    0.014730
42      driver_prev_lap_time    0.014376
46         position_prev_lap    0.011346
44    driver_best_lap_so_far    0.009638
45         driver_pace_index    0.009046
38             fuel_load_pct    0.008470
4           stint_lap_number    0.007728
39         race_progress_pct    0.007574
59         EventName_encoded    0.006985
33                 TrackTemp    0.006671
37           fuel_load_proxy    0.006202


**Beware , Sudden performance jump can be suspicious leaks, don't be happy now !!**

In [55]:
df_dry_clean = df_dry_model[df_dry_model['status_green'] == True].copy()
print(f"Clean dry data shape: {df_dry_clean.shape}")

train_dry_clean = df_dry_clean[df_dry_clean['Season'] <= 2024].copy()
test_dry_clean = df_dry_clean[df_dry_clean['Season'] == 2025].copy()

X_train_dc = train_dry_clean[feature_columns_v2_final]
y_train_dc = train_dry_clean['tyre_deg_target_v3']
X_test_dc = test_dry_clean[feature_columns_v2_final]
y_test_dc = test_dry_clean['tyre_deg_target_v3']

model_dry_clean = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
model_dry_clean.fit(X_train_dc, y_train_dc)
y_pred_dc = model_dry_clean.predict(X_test_dc)

mae_dc = mean_absolute_error(y_test_dc, y_pred_dc)
rmse_dc = np.sqrt(mean_squared_error(y_test_dc, y_pred_dc))
r2_dc = r2_score(y_test_dc, y_pred_dc)

print(f"MAE: {mae_dc:.3f}s")
print(f"RMSE: {rmse_dc:.3f}s")
print(f"R2: {r2_dc:.3f}")

importance_dc = pd.DataFrame({
    'feature': feature_columns_v2_final,
    'importance': model_dry_clean.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\nTop 20 features:")
print(importance_dc.head(20))

Clean dry data shape: (163452, 147)
MAE: 1.186s
RMSE: 2.616s
R2: 0.724

Top 20 features:
                     feature  importance
49     laps_since_sc_started    0.278566
52          prevstatus_green    0.253356
54     prevstatus_safety_car    0.084993
50    laps_since_vsc_started    0.079147
45         driver_pace_index    0.019057
42      driver_prev_lap_time    0.016442
21     warmup_laps_remaining    0.010969
56     prevstatus_vsc_ending    0.010651
38             fuel_load_pct    0.010305
44    driver_best_lap_so_far    0.009268
55            prevstatus_vsc    0.008766
4           stint_lap_number    0.008391
27         sc_recovery_decay    0.008357
20           is_warmup_phase    0.007597
25                phase_late    0.007463
28            is_restart_lap    0.007386
26  laps_since_green_resumed    0.007319
33                 TrackTemp    0.007213
32                  Rainfall    0.006923
34             WindDirection    0.006784


In [57]:
check = df_prepared.groupby(['Season', 'EventName', 'LapNumber'])['status_safety_car'].nunique()
print(f"Laps where status_safety_car varies across drivers: {(check > 1).sum()} out of {len(check)}")

check2 = df_prepared[df_prepared['status_safety_car'] == False]['laps_since_sc_started']
print(f"laps_since_sc_started when status_safety_car is False - value counts:")
print(check2.value_counts().head(10))

Laps where status_safety_car varies across drivers: 116 out of 10292
laps_since_sc_started when status_safety_car is False - value counts:
laps_since_sc_started
0    176556
Name: count, dtype: int64


In [59]:
print(f"Unique values of laps_since_sc_started in X_train_dc: {X_train_dc['laps_since_sc_started'].unique()}")
print(f"Variance: {X_train_dc['laps_since_sc_started'].var()}")

print(f"\ndf_dry_clean shape: {df_dry_clean.shape}")
print(f"Duplicate rows check (Season, EventName, Driver, LapNumber should be unique):")
dup_check = df_dry_clean.duplicated(subset=['Season','EventName','Driver','LapNumber'], keep=False)
print(f"Duplicate count: {dup_check.sum()}")

print(f"\nstatus_safety_car values when status_green==True in df_dry_clean:")
print(df_dry_clean['status_safety_car'].value_counts())

Unique values of laps_since_sc_started in X_train_dc: [ 0  1  2  5  4  6  3  7 10  9  8 12 11 13 14]
Variance: 0.2660795670493096

df_dry_clean shape: (163452, 147)
Duplicate rows check (Season, EventName, Driver, LapNumber should be unique):
Duplicate count: 0

status_safety_car values when status_green==True in df_dry_clean:
status_safety_car
False    160920
True       2532
Name: count, dtype: int64


In [60]:
truly_clean_mask = (
    (df_dry_model['status_green'] == True) &
    (df_dry_model['status_safety_car'] == False) &
    (df_dry_model['status_vsc'] == False) &
    (df_dry_model['status_yellow'] == False)
)

df_dry_clean_v2 = df_dry_model[truly_clean_mask].copy()
print(f"Before: {df_dry_model.shape[0]}, After truly-clean filter: {df_dry_clean_v2.shape[0]}")

# verify no overlap remains
print(f"status_safety_car sum: {df_dry_clean_v2['status_safety_car'].sum()}")
print(f"status_vsc sum: {df_dry_clean_v2['status_vsc'].sum()}")
print(f"status_yellow sum: {df_dry_clean_v2['status_yellow'].sum()}")

train_dc2 = df_dry_clean_v2[df_dry_clean_v2['Season'] <= 2024].copy()
test_dc2 = df_dry_clean_v2[df_dry_clean_v2['Season'] == 2025].copy()

X_train_dc2 = train_dc2[feature_columns_v2_final]
y_train_dc2 = train_dc2['tyre_deg_target_v3']
X_test_dc2 = test_dc2[feature_columns_v2_final]
y_test_dc2 = test_dc2['tyre_deg_target_v3']

model_dc2 = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
model_dc2.fit(X_train_dc2, y_train_dc2)
y_pred_dc2 = model_dc2.predict(X_test_dc2)

mae_dc2 = mean_absolute_error(y_test_dc2, y_pred_dc2)
rmse_dc2 = np.sqrt(mean_squared_error(y_test_dc2, y_pred_dc2))
r2_dc2 = r2_score(y_test_dc2, y_pred_dc2)

print(f"\nMAE: {mae_dc2:.3f}s")
print(f"RMSE: {rmse_dc2:.3f}s")
print(f"R2: {r2_dc2:.3f}")

importance_dc2 = pd.DataFrame({
    'feature': feature_columns_v2_final,
    'importance': model_dc2.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\nTop 20 features:")
print(importance_dc2.head(20))

Before: 169378, After truly-clean filter: 153514
status_safety_car sum: 0
status_vsc sum: 0
status_yellow sum: 0

MAE: 0.968s
RMSE: 1.932s
R2: 0.253

Top 20 features:
                      feature  importance
21      warmup_laps_remaining    0.076343
42       driver_prev_lap_time    0.053560
45          driver_pace_index    0.047301
38              fuel_load_pct    0.042038
11  compound_HARD_x_tracktemp    0.032695
4            stint_lap_number    0.030836
33                  TrackTemp    0.030065
59          EventName_encoded    0.028197
58               Team_encoded    0.027282
27          sc_recovery_decay    0.026844
44     driver_best_lap_so_far    0.026452
43    driver_rolling_median_3    0.025213
24                  phase_mid    0.024345
31                   Pressure    0.024265
34              WindDirection    0.024180
35                  WindSpeed    0.021841
29                    AirTemp    0.021507
51        sc_deployment_count    0.020917
26   laps_since_green_resumed    0.

In [61]:
# check correlation
corr = df_dry_clean_v2[['TyreLife', 'stint_lap_number', 'tyrelife_squared', 'tyrelife_sqrt', 'tyrelife_log1p']].corr()
print(corr)

# check full importance list, na sirf top 20
print(f"\nTyreLife rank and importance:")
print(importance_dc2[importance_dc2['feature'].str.contains('tyrelife', case=False)])
print(importance_dc2[importance_dc2['feature'] == 'TyreLife'])

                  TyreLife  stint_lap_number  tyrelife_squared  tyrelife_sqrt  \
TyreLife          1.000000          0.987424          0.937833       0.978190   
stint_lap_number  0.987424          1.000000          0.929340       0.963013   
tyrelife_squared  0.937833          0.929340          1.000000       0.852500   
tyrelife_sqrt     0.978190          0.963013          0.852500       1.000000   
tyrelife_log1p    0.924155          0.907023          0.754182       0.982862   

                  tyrelife_log1p  
TyreLife                0.924155  
stint_lap_number        0.907023  
tyrelife_squared        0.754182  
tyrelife_sqrt           0.982862  
tyrelife_log1p          1.000000  

TyreLife rank and importance:
                       feature  importance
8     compound_HARD_x_tyrelife    0.011954
0                     TyreLife    0.009867
9   compound_MEDIUM_x_tyrelife    0.008649
1             tyrelife_squared    0.008555
10    compound_SOFT_x_tyrelife    0.008416
2             

In [62]:
tyre_family = ['TyreLife', 'tyrelife_squared', 'tyrelife_sqrt', 'tyrelife_log1p',
               'stint_lap_number', 'compound_HARD_x_tyrelife', 
               'compound_MEDIUM_x_tyrelife', 'compound_SOFT_x_tyrelife',
               'SOFT_x_stint_lap', 'SOFT_x_stint_lap_sq',
               'MEDIUM_x_stint_lap', 'MEDIUM_x_stint_lap_sq',
               'HARD_x_stint_lap', 'HARD_x_stint_lap_sq']

total_tyre_importance = importance_dc2[importance_dc2['feature'].isin(tyre_family)]['importance'].sum()
print(f"Combined tyre-life-family importance: {total_tyre_importance:.4f}")

Combined tyre-life-family importance: 0.1390


## **Saving the Model**
___

In [63]:
import os
import joblib

os.makedirs('../models/tyre_degradation', exist_ok=True)

# model save
joblib.dump(model_dc2, '../models/tyre_degradation/tyre_deg_xgb_model.pkl')

# label encoders save
joblib.dump(label_encoders_dry, '../models/tyre_degradation/label_encoders.pkl')

# feature column order save (critical for live inference later)
joblib.dump(feature_columns_v2_final, '../models/tyre_degradation/feature_columns.pkl')

print("Saved:")
print(os.listdir('../models/tyre_degradation'))

Saved:
['feature_columns.pkl', 'label_encoders.pkl', 'tyre_deg_xgb_model.pkl']



# Model 2: Tyre Degradation Prediction - Final Results

## Target variable
tyre_deg_target = LapTime_seconds - stint_baseline_laptime

stint_baseline_laptime is the best (fastest) of the first 3 clean laps in a stint
(non-out-lap, green flag only). Using the single first clean lap as baseline was
tried first but caused contamination from cold tyre warm-up, since the first lap
of a fresh stint is naturally slower regardless of degradation. Taking the best of
the first 3 avoids this.

Out-laps are excluded from the target entirely.

## Data scope
Training done only on truly clean laps: status_green == True AND status_safety_car,
status_vsc, status_yellow all False. This was necessary because these status flags
are not mutually exclusive in the base table (a lap can have both status_green and
status_safety_car true if the safety car deployed mid-lap), so a plain status_green
filter alone still let safety car and VSC contaminated laps into training. Those
laps have lap times heavily distorted by regulation-based slowdown, unrelated to
tyre wear, and dominated the model if left in.

Dry compounds (HARD, MEDIUM, SOFT) modeled separately from wet compounds
(INTERMEDIATE, WET), since rain variability in lap time is a much larger and
different phenomenon than thermal tyre degradation. This notebook covers the dry
compound model. Wet compound modeling is a possible future extension, not done here.

## New features added beyond the base prepare_race_features() set
- warmup_laps_remaining, is_warmup_phase, stint_phase (warmup/early/mid/late):
  separates cold tyre warm-up behavior from genuine degradation
- laps_since_green_resumed, sc_recovery_decay, is_restart_lap: captures the lingering
  effect of a safety car or VSC period even after it ends (tyre temperature and
  pressure take a few laps to recover, not an instant reset)
- compound x stint_lap_number and compound x stint_lap_number_squared interactions,
  per compound (SOFT, MEDIUM, HARD): lets the model fit a different degradation
  curve shape per compound type

## Model
XGBoost, same baseline config as Model 1 (n_estimators=300, max_depth=6,
learning_rate=0.05, subsample=0.8, colsample_bytree=0.8). No hyperparameter tuning
done, consistent with Model 1's finding that tuning didn't help and feature
engineering was the real lever.

## Result (dry compounds, 2025 holdout)
MAE: 0.968s
RMSE: 1.932s
R2: 0.253

## Interpreting the R2
R2 is much lower than Model 1's 0.896, but this is expected and not a modeling
flaw. Once safety car, VSC, yellow flag, and out-laps are removed, what remains
is a genuinely small and noisy signal: tyre degradation on a clean lap is usually
a fraction of a second to a couple of seconds, competing against driver-to-driver
pace variance, track-to-track differences, and residual fuel burn effects that
are not fully separable from degradation in a single lap's time. Real tyre models
in motorsport engineering deal with the same difficulty. MAE under 1 second is
the more meaningful number here, since it reflects that predictions are close in
absolute terms even though the variance explained is modest.

## Feature importance check
TyreLife, tyrelife_squared, tyrelife_sqrt, stint_lap_number, and the compound x
tyrelife interaction terms are highly correlated with each other (TyreLife and
stint_lap_number alone correlate at 0.987), so XGBoost splits importance across
them rather than concentrating it in one. Summed together, this tyre-life feature
family accounts for about 13.9% of total importance, more than any single other
feature, confirming the model is genuinely picking up a tyre-related signal
rather than something spurious.

## Known limitations carried forward
- SC-start / rolling-start races still have the same v1 limitation as Model 1
  (no reliable pre-race signal for race-start-type)
- Wet compound races are excluded from this model, would need separate treatment
- This is a per-lap degradation delta model, not a per-stint degradation rate
  model, so it does not directly output a strategy-ready "tyre falls off at lap X"
  answer on its own, that would need to be derived from the predicted curve across
  a simulated stint

## Saved artifacts
models/tyre_degradation/tyre_deg_xgb_model.pkl
models/tyre_degradation/label_encoders.pkl
models/tyre_degradation/feature_columns.pkl